# CatBoost + Optuna (Kaggle Irrigation Need)

This notebook is CatBoost-only and follows the same structure as the LightGBM notebook:

1. Build multiple CatBoost configurations (at least 3 hyperparameter sets).
2. Compare cross-validated **balanced accuracy**.
3. Tune CatBoost with Optuna.
4. Train the best CatBoost model on full training data and create a Kaggle submission.

In [ ]:
%pip install catboost optuna -q

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, balanced_accuracy_score, accuracy_score

from catboost import CatBoostClassifier
import optuna

In [ ]:
# Load data
train = pd.read_csv("../Kaggle/train.csv")
test = pd.read_csv("../Kaggle/test.csv")

TARGET_COL = "Irrigation_Need"
ID_COL = "id"

feature_cols = [c for c in train.columns if c not in [ID_COL, TARGET_COL]]

X_train_raw = train[feature_cols].copy()
X_test_raw = test[feature_cols].copy()
y_raw = train[TARGET_COL].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Target distribution:")
print(y_raw.value_counts(normalize=True).round(4))

In [ ]:
# Encode target
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

print("Label mapping:")
for cls, enc in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"  {cls} -> {enc}")

# Use one-hot encoding to keep preprocessing consistent with other notebooks
all_features = pd.concat([X_train_raw, X_test_raw], axis=0)
all_features = pd.get_dummies(all_features, drop_first=False)

X = all_features.iloc[: len(X_train_raw)].copy()
X_test = all_features.iloc[len(X_train_raw):].copy()

print("Encoded train feature shape:", X.shape)
print("Encoded test feature shape:", X_test.shape)

## CatBoost Formulas and Hyperparameter Intuition

For multiclass boosting rounds \(t = 1, \dots, T\):

- Additive model update:
  \[
  F_k^{(t)}(x) = F_k^{(t-1)}(x) + \eta \cdot h_{k,t}(x)
  \]

- Multiclass probability (softmax):
  \[
  p_k(x) = \frac{\exp(F_k(x))}{\sum_{j=1}^{K} \exp(F_j(x))}
  \]

- Competition-relevant evaluation metric:
  \[
  \text{Balanced Accuracy} = \frac{1}{K}\sum_{k=1}^{K} \text{Recall}_k
  \]

Main CatBoost hyperparameters tested here: `iterations`, `learning_rate`, `depth`, `l2_leaf_reg`, `bagging_temperature`, and `random_strength`.

In [ ]:
# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

base_params = {
    "loss_function": "MultiClass",
    "eval_metric": "MultiClass",
    "random_seed": 42,
    "verbose": 0,
}

# At least 3 distinct parameter sets to show hyperparameter impact
manual_param_sets = {
    "set_1_baseline": {
        "iterations": 400,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 3.0,
        "bagging_temperature": 1.0,
        "random_strength": 1.0,
    },
    "set_2_more_regularized": {
        "iterations": 700,
        "learning_rate": 0.03,
        "depth": 7,
        "l2_leaf_reg": 8.0,
        "bagging_temperature": 2.0,
        "random_strength": 2.0,
    },
    "set_3_aggressive": {
        "iterations": 1000,
        "learning_rate": 0.02,
        "depth": 9,
        "l2_leaf_reg": 2.0,
        "bagging_temperature": 0.3,
        "random_strength": 0.5,
    },
}

manual_results = []
for name, p in manual_param_sets.items():
    params = base_params.copy()
    params.update(p)

    model = CatBoostClassifier(**params)
    scores = cross_val_score(model, X, y, cv=cv, scoring="balanced_accuracy", n_jobs=-1)

    manual_results.append(
        {
            "config": name,
            "cv_balanced_accuracy_mean": scores.mean(),
            "cv_balanced_accuracy_std": scores.std(),
        }
    )

manual_results_df = pd.DataFrame(manual_results).sort_values("cv_balanced_accuracy_mean", ascending=False)
manual_results_df

In [ ]:
def objective(trial):
    params = base_params.copy()
    params.update(
        {
            "iterations": trial.suggest_int("iterations", 300, 1400),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "depth": trial.suggest_int("depth", 4, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
            "random_strength": trial.suggest_float("random_strength", 0.0, 5.0),
            "border_count": trial.suggest_int("border_count", 32, 255),
        }
    )

    model = CatBoostClassifier(**params)
    scores = cross_val_score(model, X, y, cv=cv, scoring="balanced_accuracy", n_jobs=-1)
    return scores.mean()

In [ ]:
# Run Optuna tuning
N_TRIALS = 35  # increase to 75-150 if runtime allows

study = optuna.create_study(direction="maximize", study_name="catboost_irrigation_optuna")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_manual = manual_results_df.iloc[0]["cv_balanced_accuracy_mean"]
print("Best manual-set CV Balanced Accuracy:", round(float(best_manual), 5))
print("Best Optuna CV Balanced Accuracy:", round(study.best_value, 5))
print("Improvement over best manual set:", round(study.best_value - float(best_manual), 5))

print("\nBest Optuna params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Fit tuned CatBoost model on full training set
best_params = study.best_params.copy()
best_params.update(
    {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "random_seed": 42,
        "verbose": 0,
    }
)

best_model = CatBoostClassifier(**best_params)
best_model.fit(X, y)

# Quick training-set check (not leaderboard metric)
train_pred = best_model.predict(X).reshape(-1)
print("Train accuracy:", round(accuracy_score(y, train_pred), 5))
print("Train balanced accuracy:", round(balanced_accuracy_score(y, train_pred), 5))
print("\nTrain classification report:")
print(classification_report(y, train_pred, target_names=label_encoder.classes_))

In [ ]:
# Predict test and create submission
test_pred_encoded = best_model.predict(X_test).reshape(-1).astype(int)
test_pred_labels = label_encoder.inverse_transform(test_pred_encoded)

submission = pd.DataFrame(
    {
        ID_COL: test[ID_COL],
        TARGET_COL: test_pred_labels,
    }
)

submission_path = "../Kaggle/submission_catboost_optuna.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
submission.head()

## Notes on hyperparameter effects

After running `manual_results_df` and Optuna:

- Compare the 3 manual sets to describe how `depth`, `learning_rate`, and `iterations` changed balanced accuracy.
- Report whether stronger regularization (`l2_leaf_reg`, `bagging_temperature`, `random_strength`) improved generalization.
- Record how much Optuna improved over your best manual set.
- Upload `submission_catboost_optuna.csv` to Kaggle and log the leaderboard score.